In [ ]:
# Lecture 26 — Final Project: Starter Notebook
# Spark 101 · Module 8 · Lecture 26
#
# Import this file into a Databricks workspace as a notebook.
# One question is fully worked as a template; the rest are TODOs — make them yours.
# Databricks recognizes `# COMMAND ----------` as a cell boundary on import.

In [ ]:
print("Spark version:", spark.version)
# Databricks gives you `spark` for free — no SparkSession.builder needed.

In [ ]:
# --- Load the dataset (built-in Databricks sample — no upload) --------------
# Verify the exact table path in your workspace; the classic sample is:
trips = spark.read.table("samples.nyctaxi.trips")

# Explore FIRST — always (L14).
trips.printSchema()
trips.show(5)
print("rows:", trips.count())

In [ ]:
# --- Cache the base DataFrame you reuse across every question (L23) ---------
# Optionally trim to the columns you'll use before caching (L22: select early).
from pyspark.sql.functions import col, hour, to_date, avg, sum as _sum, count, round as _round

base = trips.select(
    "tpep_pickup_datetime",
    "trip_distance",
    "fare_amount",
    "pickup_zip",
    "dropoff_zip",
).cache()
base.count()   # materialize the cache

In [ ]:
# ============================================================================
# WORKED QUESTION (template): revenue by pickup zip — top 10
# ============================================================================
revenue_by_zip = (
    base
    .filter(col("fare_amount") > 0)                 # narrow: clean first (L16)
    .groupBy("pickup_zip")                          # wide: shuffle (L22)
    .agg(
        count("*").alias("trips"),
        _round(_sum("fare_amount"), 2).alias("total_fare"),
        _round(avg("fare_amount"), 2).alias("avg_fare"),
    )
    .orderBy(col("total_fare").desc())
)
revenue_by_zip.show(10)   # SAFE — never collect() the whole thing (L25)

In [ ]:
# --- Write the result as Parquet (L19–L20) ---------------------------------
revenue_by_zip.write.mode("overwrite").parquet("file:/tmp/spark101_project/revenue_by_zip")

In [ ]:
# ============================================================================
# TODO — Question 2: trips and revenue by HOUR of day. When is the peak?
#   hint: hour("tpep_pickup_datetime"), groupBy, agg, orderBy
# ============================================================================

In [ ]:
# ============================================================================
# TODO — Question 3: fare per mile distribution. Find the outliers.
#   hint: fare_amount / trip_distance (guard against distance = 0), summary()
# ============================================================================

In [ ]:
# ============================================================================
# TODO — Question 4: daily trip volume over the date range.
#   hint: to_date("tpep_pickup_datetime"), groupBy, count, orderBy
# ============================================================================

In [ ]:
# ============================================================================
# TODO — Question 5 (your own): one thing you're curious about in this data.
# ============================================================================

In [ ]:
# --- When you're done: release the cache -----------------------------------
base.unpersist()

In [ ]:
# WORKED EXAMPLE 2: trips and revenue by hour of day.
from pyspark.sql.functions import hour, count, sum as _sum, round as _round
by_hour = (base.withColumn("hr", hour("tpep_pickup_datetime"))
               .groupBy("hr")
               .agg(count("*").alias("trips"), _round(_sum("fare_amount"), 0).alias("revenue"))
               .orderBy("hr"))
by_hour.show(24)

In [ ]:
# WORKED EXAMPLE 3: daily trip volume over the date range.
from pyspark.sql.functions import to_date
(base.withColumn("day", to_date("tpep_pickup_datetime"))
     .groupBy("day").count().orderBy("day").show(10))

In [ ]:
# WORKED EXAMPLE 4: top-10 zips by trip count, ranked with a window function.
from pyspark.sql.functions import count, col, row_number
from pyspark.sql.window import Window
w = Window.orderBy(col("trips").desc())
ranked = (base.groupBy("pickup_zip").agg(count("*").alias("trips"))
              .withColumn("rank", row_number().over(w)))
ranked.filter("rank <= 10").show()

In [ ]:
# Deliverable checklist:
#   [ ] reads the data, explores the schema
#   [ ] answers 3–5 questions with select/filter/groupBy/join or SQL
#   [ ] writes results as Parquet
#   [ ] no collect() on full data; no needless shuffles; cached the reused base
#   [ ] a colleague can read it top to bottom
# Ship it.